# C4.5 from scratch

Uses Gain Ratio to choose categorical attributes; this compact teaching implementation does not include all production C4.5 features such as pruning and continuous-feature thresholds.

**No scikit-learn is used.** Requirements: `pandas`, `openpyxl`, `numpy`, `matplotlib`. Put `data.xlsx` beside the notebook. Change `TARGET` if the last column is not your class label.

In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
from math import log2

FILE = "data.xlsx"
df = pd.read_excel(FILE).dropna().copy()
print(df.head())
print("Columns:", list(df.columns))

TARGET = df.columns[-1]  # change if needed
MAX_DEPTH = 4

def entropy(y):
    counts = Counter(y)
    n = len(y)
    return -sum((v/n)*log2(v/n) for v in counts.values() if v)

def gain_ratio(data, col, target):
    base = entropy(data[target])
    n = len(data)
    groups = list(data.groupby(col))
    rem = sum(len(g)/n * entropy(g[target]) for _, g in groups)
    gain = base - rem
    split_info = -sum((len(g)/n)*log2(len(g)/n) for _, g in groups if len(g))
    return gain / split_info if split_info else 0

def c45_tree(data, features, target, depth=0):
    y = data[target]
    if y.nunique() == 1 or depth >= MAX_DEPTH or not features:
        return y.mode().iloc[0]
    ratios = {f: gain_ratio(data, f, target) for f in features}
    best = max(ratios, key=ratios.get)
    if ratios[best] <= 0:
        return y.mode().iloc[0]
    node = {"feature": best, "branches": {}, "fallback": y.mode().iloc[0]}
    for val, group in data.groupby(best):
        node["branches"][val] = c45_tree(group, [f for f in features if f != best], target, depth+1)
    return node

def predict_one(tree, row):
    if not isinstance(tree, dict): return tree
    branch = tree["branches"].get(row[tree["feature"]], tree["fallback"])
    return predict_one(branch, row)

features = [c for c in df.columns if c != TARGET]
tree = c45_tree(df, features, TARGET)
pred = df.apply(lambda r: predict_one(tree, r), axis=1)
print("C4.5 tree:", tree)
print("Training accuracy:", round((pred == df[TARGET]).mean(), 3))
pd.Series(pred).value_counts().plot(kind="bar", title="C4.5 predicted class counts")
plt.xlabel("Predicted class"); plt.ylabel("Count"); plt.tight_layout(); plt.show()
